In [ ]:
import re, json
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, TimeDistributed, Dense
from tensorflow.keras.models import Model

In [ ]:
raw = load_dataset("ai4privacy/pii-masking-200k", data_files=["english_pii_43k.jsonl"])
# if load_dataset returns a dict with “train” split:
dataset = raw["train"] if isinstance(raw, dict) else raw

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:45: UserWarning: The cache_dir for this dataset is /root/.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [DBFS].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/dbfs/']
  warnings.warn(warning_message)
/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:14: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


In [ ]:
raw

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 43501
    })
})

In [ ]:
# determine which token/label fields exist
cols = set(dataset.column_names)
if "tokenised_text" in cols and "bio_labels" in cols:
    token_seqs = dataset["tokenised_text"]
    label_seqs = dataset["bio_labels"]
elif "mbert_text_tokens" in cols and "mbert_bio_labels" in cols:
    token_seqs = dataset["mbert_text_tokens"]
    label_seqs = dataset["mbert_bio_labels"]
else:
    raise ValueError(
        f"Expected one of (tokenised_text, bio_labels) or "
        f"(mbert_text_tokens, mbert_bio_labels) in columns: {dataset.column_names}"
    )

if not token_seqs:
    raise ValueError("No token sequences found after filtering – check your filter() call.")

In [ ]:
# a) tokens → index (lowercasing subword tokens is fine here)
vocab = {tok.lower() for seq in token_seqs for tok in seq}
token2idx = {tok: i+2 for i, tok in enumerate(sorted(vocab))}
token2idx["PAD"] = 0
token2idx["OOV"] = 1

# b) tags → index
tags = {tag for seq in label_seqs for tag in seq}
tag2idx = {tag: i for i, tag in enumerate(sorted(tags))}
idx2tag = {i: t for t, i in tag2idx.items()}

# c) max sequence length
MAX_LEN = max(len(seq) for seq in token_seqs)
print(f"Using MAX_LEN = {MAX_LEN}, vocab_size = {len(token2idx)}, n_tags = {len(tag2idx)}")

Using MAX_LEN = 783, vocab_size = 15701, n_tags = 113


In [ ]:
def encode_and_pad(seqs, mapping, pad_value, max_len):
    out = []
    for seq in seqs:
        idxs = [mapping.get(tok.lower(), pad_value) for tok in seq]
        if len(idxs) < max_len:
            idxs += [pad_value] * (max_len - len(idxs))
        else:
            idxs = idxs[:max_len]
        out.append(idxs)
    return np.array(out)

X = encode_and_pad(token_seqs, token2idx, pad_value=token2idx["PAD"], max_len=MAX_LEN)
y = encode_and_pad(label_seqs, tag2idx, pad_value=tag2idx["O"], max_len=MAX_LEN)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
input_layer = Input(shape=(MAX_LEN,), dtype="int32")
emb = Embedding(
    input_dim=len(token2idx),
    output_dim=128,
    mask_zero=True
)(input_layer)
lstm = Bidirectional(LSTM(64, return_sequences=True))(emb)
out  = TimeDistributed(Dense(len(tag2idx), activation="softmax"))(lstm)

model = Model(inputs=input_layer, outputs=out)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 783)]             0         
                                                                 
 embedding_1 (Embedding)     (None, 783, 128)          2009728   
                                                                 
 bidirectional_1 (Bidirecti  (None, 783, 128)          98816     
 onal)                                                           
                                                                 
 time_distributed_1 (TimeDi  (None, 783, 113)          14577     
 stributed)                                                      
                                                                 
Total params: 2123121 (8.10 MB)
Trainable params: 2123121 (8.10 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
# y needs an extra dimension for sparse loss
y_train_exp = np.expand_dims(y_train, -1)

model.fit(
    X_train, y_train_exp,
    validation_split=0.1,
    batch_size=32,
    epochs=3
)

Epoch 1/3
145/979 [===>..........................] - ETA: 8:28 - loss: 0.5203 

*** WARNING: max output size exceeded, skipping output. ***

979/979 [==============================] - 606s 619ms/step - loss: 4.2049e-05 - accuracy: 1.0000 - val_loss: 3.1667e-05 - val_accuracy: 1.0000


In [ ]:
model.save("machi_pii_scrubber.h5")
with open("token2idx.json", "w") as f:
    json.dump(token2idx, f)
with open("tag2idx.json", "w") as f:
    json.dump(tag2idx, f)

/databricks/python/lib/python3.10/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
